In [11]:
# -------------------------------------------------
# STEP 1 — Import libraries
# -------------------------------------------------

import sqlite3
import pandas as pd
from pathlib import Path


# -------------------------------------------------
# STEP 2 — Connect to NordicFlow SQLite database
# -------------------------------------------------

db_path = Path("../data/NordicFlow_ERP.db")

conn = sqlite3.connect(db_path)


# -------------------------------------------------
# STEP 3 — Load required ERP tables
# -------------------------------------------------

# Production transactions
production = pd.read_sql_query(
    "SELECT * FROM production_orders",
    conn
)

# Material master provides finished-product names
material = pd.read_sql_query(
    "SELECT * FROM material_master",
    conn
)

# Plant master provides plant descriptions
plant = pd.read_sql_query(
    "SELECT * FROM plant_master",
    conn
)

print("Production orders:", len(production))
print("Materials:", len(material))
print("Plants:", len(plant))

# -------------------------------------------------
# STEP 4 — Prepare data types
# -------------------------------------------------

# Convert production dates to Python datetime format
date_columns = [
    "Order_Creation_Date",
    "Planned_Start_Date",
    "Actual_Start_Date",
    "Planned_End_Date",
    "Actual_End_Date"
]

for column in date_columns:
    production[column] = pd.to_datetime(
        production[column],
        dayfirst=True,
        errors="coerce"
    )


# Convert material shortage flag from ERP text into 1/0
production["Material_Shortage_Flag_Calc"] = (
    production["Material_Shortage_Flag"]
    .astype(str)
    .str.strip()
    .str.upper()
    .map({
        "TRUE": 1,
        "FALSE": 0,
        "1": 1,
        "0": 0
    })
)

print(production.dtypes)

# -------------------------------------------------
# STEP 5 — Calculate production KPIs
# -------------------------------------------------

total_orders = len(production)

# Orders completed on or before planned completion
on_time_orders = (
    production["Completion_Delay_Days"] <= 0
).sum()

# Orders completed late
delayed_orders = (
    production["Completion_Delay_Days"] > 0
).sum()

# Production schedule adherence %
production_otd = (
    on_time_orders / total_orders * 100
)

# Overall production quantity attainment %
production_attainment = (
    production["Actual_Qty"].sum()
    / production["Planned_Qty"].sum()
    * 100
)

# Material-shortage affected orders
shortage_orders = (
    production["Material_Shortage_Flag_Calc"].sum()
)

production_kpis = pd.DataFrame({
    "KPI": [
        "Production Orders",
        "On-Time Orders",
        "Delayed Orders",
        "Production On-Time %",
        "Production Attainment %",
        "Material Shortage Orders"
    ],
    "Value": [
        total_orders,
        on_time_orders,
        delayed_orders,
        round(production_otd, 1),
        round(production_attainment, 1),
        int(shortage_orders)
    ]
})

production_kpis

# -------------------------------------------------
# STEP 6 — Analyse production delay reasons
# -------------------------------------------------

delay_analysis = (
    production
    .groupby("Delay_Reason", as_index=False)
    .agg(
        Production_Orders=("Production_Order_ID", "count"),
        Average_Delay_Days=("Completion_Delay_Days", "mean"),
        Maximum_Delay_Days=("Completion_Delay_Days", "max")
    )
)

delay_analysis["Average_Delay_Days"] = (
    delay_analysis["Average_Delay_Days"].round(1)
)

delay_analysis.sort_values(
    "Production_Orders",
    ascending=False
)

# -------------------------------------------------
# STEP 7 — Measure impact of material shortages
# -------------------------------------------------

production["Shortage_Status"] = production[
    "Material_Shortage_Flag_Calc"
].map({
    1: "Material Shortage",
    0: "No Material Shortage"
})

shortage_impact = (
    production
    .groupby("Shortage_Status", as_index=False)
    .agg(
        Production_Orders=("Production_Order_ID", "count"),
        Average_Completion_Delay_Days=(
            "Completion_Delay_Days",
            "mean"
        ),
        Average_Planned_Duration_Days=(
            "Planned_Duration_Days",
            "mean"
        ),
        Average_Actual_Duration_Days=(
            "Actual_Duration_Days",
            "mean"
        )
    )
)

shortage_impact[
    "Average_Completion_Delay_Days"
] = shortage_impact[
    "Average_Completion_Delay_Days"
].round(1)

shortage_impact

# -------------------------------------------------
# STEP 8 — Compare production performance by plant
# -------------------------------------------------

plant_performance = (
    production
    .groupby("Plant_ID", as_index=False)
    .agg(
        Production_Orders=("Production_Order_ID", "count"),
        Average_Delay_Days=("Completion_Delay_Days", "mean"),
        Planned_Qty=("Planned_Qty", "sum"),
        Actual_Qty=("Actual_Qty", "sum")
    )
)

# Calculate production attainment by plant
plant_performance["Production_Attainment_Pct"] = (
    plant_performance["Actual_Qty"]
    / plant_performance["Planned_Qty"]
    * 100
)

# Calculate plant-specific on-time %
plant_otd = (
    production
    .assign(
        On_Time=production["Completion_Delay_Days"] <= 0
    )
    .groupby("Plant_ID")["On_Time"]
    .mean()
    .mul(100)
    .reset_index(name="Production_On_Time_Pct")
)

plant_performance = plant_performance.merge(
    plant_otd,
    on="Plant_ID",
    how="left"
)

# Add plant names
plant_performance = plant_performance.merge(
    plant[
        [
            "Plant_ID",
            "Plant_Name"
        ]
    ],
    on="Plant_ID",
    how="left"
)

plant_performance[
    [
        "Plant_ID",
        "Plant_Name",
        "Production_Orders",
        "Average_Delay_Days",
        "Production_On_Time_Pct",
        "Production_Attainment_Pct"
    ]
].round(1)

# -------------------------------------------------
# STEP 9 — Analyse production performance by product
# -------------------------------------------------

product_performance = (
    production
    .groupby("Finished_Product_ID", as_index=False)
    .agg(
        Production_Orders=("Production_Order_ID", "count"),
        Average_Delay_Days=("Completion_Delay_Days", "mean"),
        Maximum_Delay_Days=("Completion_Delay_Days", "max"),
        Planned_Qty=("Planned_Qty", "sum"),
        Actual_Qty=("Actual_Qty", "sum")
    )
)

# Add product names from material master
product_performance = product_performance.merge(
    material[
        [
            "Material_ID",
            "Material_Name"
        ]
    ],
    left_on="Finished_Product_ID",
    right_on="Material_ID",
    how="left"
)

product_performance.sort_values(
    "Average_Delay_Days",
    ascending=False
)

# -------------------------------------------------
# STEP 10 — Create production exception report
# -------------------------------------------------

production_exceptions = production[
    (
        production["Completion_Delay_Days"] > 0
    )
    |
    (
        production["Material_Shortage_Flag_Calc"] == 1
    )
].copy()

production_exceptions = production_exceptions[
    [
        "Production_Order_ID",
        "Finished_Product_ID",
        "Plant_ID",
        "Production_Priority",
        "Completion_Delay_Days",
        "Delay_Reason",
        "Material_Shortage_Flag_Calc",
        "Planned_Qty",
        "Actual_Qty"
    ]
].sort_values(
    "Completion_Delay_Days",
    ascending=False
)

production_exceptions

# -------------------------------------------------
# STEP 11 — Export production outputs
# -------------------------------------------------

output_path = Path("../outputs")

output_path.mkdir(
    parents=True,
    exist_ok=True
)

production_kpis.to_csv(
    output_path / "production_kpis.csv",
    index=False
)

delay_analysis.to_csv(
    output_path / "production_delay_analysis.csv",
    index=False
)

shortage_impact.to_csv(
    output_path / "material_shortage_impact.csv",
    index=False
)

plant_performance.to_csv(
    output_path / "plant_production_performance.csv",
    index=False
)

product_performance.to_csv(
    output_path / "product_production_performance.csv",
    index=False
)

production_exceptions.to_csv(
    output_path / "production_exceptions.csv",
    index=False
)

print("Production analytics exported successfully.")

,Production_Order_ID,Finished_Product_ID,Plant_ID,Production_Priority,Completion_Delay_Days,Delay_Reason,Material_Shortage_Flag_Calc,Planned_Qty,Actual_Qty
8,PRD-7009,FG-3001,PLT-1000,Urgent,7,Supplier Delay,1,6,6
1,PRD-7002,FG-1002,PLT-1000,High,6,Material Shortage,1,8,8
3,PRD-7004,FG-3001,PLT-1000,Urgent,6,Supplier Delay,1,5,5
5,PRD-7006,FG-1001,PLT-1000,Urgent,6,Quality Hold,1,15,15
10,PRD-7011,FG-1001,PLT-1000,High,6,Material Shortage,1,14,14
13,PRD-7014,FG-3001,PLT-1000,Urgent,6,Supplier Delay,1,5,5
6,PRD-7007,FG-1002,PLT-1000,High,5,Material Shortage,1,10,9
11,PRD-7012,FG-1002,PLT-1000,High,5,Material Shortage,1,9,9
0,PRD-7001,FG-1001,PLT-1000,High,4,Material Shortage,1,12,12
4,PRD-7005,FG-4001,PLT-2000,Normal,1,Capacity Constraint,0,10,10
